# задание 1. построение дискретной языковой модели на основе цепей Маркова

используем аппарат дискретных цепей Маркова для стохастической генерации текста

марковское свойство: условное распределение будущего состояния зависит исключительно от текущего состояния и не зависит от предыстории:
$$P(X_{n+1} = x_{n+1} \mid X_n = x_n, X_{n-1} = x_{n-1}, \dots, X_0 = x_0) = P(X_{n+1} = x_{n+1} \mid X_n = x_n)$$

в данной работе мы рассматриваем однородную цепь Маркова, где вероятности перехода не зависят от шага n. основной опертор такой модели: переходная матрица $P = (p_{ij})$. в ней каждый элемент задает вероятность перехода из состояния $i$ в состояние $j$:
$$p_{ij} = P(X_{n+1} = j \mid X_n = i), \quad \sum_{j} p_{ij} = 1$$

матрица является стохастической. паспределение вероятностей в момент времени n находится через начальное распределение $\pi^{(0)}$ и n-ю степень матрицы переходов: $\pi^{(n)} = \pi^{(0)} P^n$, это напрямую следует из уравнения Колмогорова-Чепмена:
$$p_{ij}^{(n+m)} = \sum_{k} p_{ik}^{(n)} p_{kj}^{(m)}$$
(так мы декомпозировуепм многошаговые переходы через промежуточные состояния k)

пространство состояний S в нашей текстовой модели конечно (определяется словарем). состояния i и j - сообщающмеся, если j достижимо из i, а i достижимо из j. если все состояния сообщаются, то цепь неразложимая

стационарное распределение - вектора вероятностей $\pi$, удовлетворяющего уравнению баланса:
$$\pi P = \pi, \quad \sum_{i} \pi_i = 1$$

эргодическая теорема: если цепь эргодическая (неразложимая и не периодическая), то существует единственное стационарное распределение, совпадающее с предельным распределением при $n \to \infty$. физический смысл $\pi_i$ - величина, обратная среднему времени первого возврата в вершину $i$ ($\mu_i = 1/\pi_i$). если предельные вероятности не зависят от начального состояния, цепь называют сильно эргодической

In [9]:
import numpy as np
import pandas as pd
import re
import random
from collections import defaultdict, Counter

загрузка и подготовка датасета

In [10]:
from google.colab import drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/стат_для_мо_labs/dostoyevskii-pin.txt'
with open(file_path, 'r', encoding='cp1251') as f:
        raw_text = f.read()

def clean_and_tokenize(text):
    # приведение к нижнему регистру для нормализации пространства состояний
    text = text.lower()
    # удаление неинформативных символов, сохраняя базовую пунктуацию конца предложений как маркеры структуры
    text = re.sub(r'[^а-яёa-z\s\.\!\?]', '', text)
    # изолируем знаки препинания, чтобы они выступали как отдельные состояния цепи
    text = re.sub(r'([\.\!\?])', r' \1 ', text)
    tokens = [token for token in text.split() if token]
    return tokens

tokens = clean_and_tokenize(raw_text)
print(f"общее число токенов в обработанном корпусе: {len(tokens)}")
print(f"пример первых 15 токенов: {tokens[:15]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
общее число токенов в обработанном корпусе: 210666
пример первых 15 токенов: ['спасибо', 'что', 'скачали', 'книгу', 'в', 'бесплатной', 'электронной', 'библиотеке', 'royallib', '.', 'ru', 'httproyallib', '.', 'ru', 'все']


строим переходную матрицу

In [11]:
def build_transition_matrix(tokens):
    # подсчет абсолютных частот переходов i -> j
    transition_counts = defaultdict(Counter)
    for i in range(len(tokens) - 1):
        current_word = tokens[i]
        next_word = tokens[i+1]
        transition_counts[current_word][next_word] += 1

    # преобразование частот в вероятности (нормировка строк)
    matrix = defaultdict(dict)
    for current_word, next_words in transition_counts.items():
        total_transitions = sum(next_words.values())
        for next_word, count in next_words.items():
            # оценка максимального правдоподобия для вероятности p_ij
            matrix[current_word][next_word] = count / total_transitions

    return matrix

P_matrix = build_transition_matrix(tokens)
vocabulary_size = len(P_matrix)
print(f"мощность пространства состояний (размер словаря): {vocabulary_size}")

мощность пространства состояний (размер словаря): 30349


генерация и оценка адекватности модели

In [12]:
def generate_phrase(matrix, start_word=None, max_length=20):
    if not start_word or start_word not in matrix:
        start_word = random.choice(list(matrix.keys()))

    phrase = [start_word]
    current = start_word

    for _ in range(max_length - 1):
        if current not in matrix or not matrix[current]:
            break # попадание в поглощающее состояние (изолированная вершина графа)

        # семплирование следующего состояния на основе распределения строки текущего состояния
        next_states = list(matrix[current].keys())
        probabilities = list(matrix[current].values())

        current = random.choices(next_states, weights=probabilities)[0]
        phrase.append(current)

        if current in ['.', '!', '?']:
            break # естественное завершение предложения

    return " ".join(phrase)

# генерация выборки фраз для экспертной оценки
generated_sentences = [generate_phrase(P_matrix) for _ in range(30)]

# наивный подсчет адекватности (критерий: связность, отсутствие синтаксического коллапса)
# пройдемся автоматическим фильтром на минимальную длину и структуру, имитируя базовую разметку
adequate_count = 0
print("--- сгенерированные фразы и их верификация ---")
for idx, sent in enumerate(generated_sentences):
    # простой лингвистический предикат: фраза длиннее 4 слов и завершается точкой - мэйби адекватен
    words_count = len(sent.split())
    is_adequate = words_count > 4 and sent[-1] in ['.', '!', '?']
    if is_adequate:
        adequate_count += 1
    print(f"[{'OK' if is_adequate else 'FAIL'}] #{idx+1}: {sent}")

adequacy_rate = adequate_count / len(generated_sentences)
print(f"\nэмпирический показатель адекватности модели: {adequacy_rate:.2%}")

--- сгенерированные фразы и их верификация ---
[OK] #1: старушечий что знает для него обвинений и допускал тут я тебя отчетов ?
[OK] #2: надеясь что бы загладилось неизмеримою сравнительно с улицы и решительным голосом да наплевать !
[FAIL] #3: кланялись матушка пора .
[FAIL] #4: подали петру петровичу .
[FAIL] #5: хитрецы .
[FAIL] #6: помещение его это уж говорите дальше к малой азии на своей сознаются дас бывалос неоднократнос проревел опять опять кинулся к
[OK] #7: летами может быть заметил в его .
[FAIL] #8: фанатизм !
[FAIL] #9: поучался у коломенских .
[OK] #10: предназначаются им быть действительно о ком я думал что не расспрашивала на то слишком на сцене в петербург он .
[FAIL] #11: выступил на сторону а что вам тогда отворилась настежь и минут и говорить что он вошел разбудил его нигилизм зимние
[OK] #12: отогнал ее положение ее еще тебе надобно же касается до того уже не прерывая молчание ?
[FAIL] #13: сердечно велось и приняться за руку он потому даже может быть в сибирке и 

# задание № 2. анализ систем массового обслуживания с помощью непрерывных цепей Маркова

отличие от дискретных цепей: в непрерывных цепях Маркова переходы между состояниями могут происходить в любой непрерывный момент времени $t \ge 0$. поведение такой системы полностью описывается инфинитезимальной матрицей (генератором) $Q = (q_{ij})$, у неё компоненты отражают интенсивности переходов:
* $i \neq j$: $q_{ij} \ge 0$ - интенсивность перехода из $i$ в $j$
* $i = j$: $q_{ii} = -\sum_{j \neq i} q_{ij}$ - суммарная интенсивность ухода из состояния $i$

динамика распределения вероятностей состояний $P_i(t)$ во времени подчиняется системе дифференциальных уравнений(уравнениям Колмогорова-Феллера):
$$\frac{dP_j(t)}{dt} = \sum_{i} P_i(t) q_{ij}$$
в матричной форме: $\frac{dP(t)}{dt} = P(t)Q$

для эргодических цепей при $t \to \infty$ производные стремятся к нулю ($\frac{dP_j(t)}{dt} \to 0$), и система переходит в стационарный режим. согласно эргодической теореме для непрерывных цепей, стационарные вероятности $\mathbf{p}$ однозначно находятся из алгебраической системы:
$$\mathbf{p} Q = \mathbf{0}, \quad \sum_{i} p_i = 1$$

мы рассмотрим систему массового обслуживания (СМО), где состояние - текущая длина очереди k. Переход $k \to k+1$ происходит с интенсивностью поступления заявок $\lambda$, а переход $k \to k-1$ — с интенсивностью обслуживания $\mu$

генерация датасета очереди (лог СМО)

In [13]:
# эмулируем работу сервера: фиксируем моменты времени и длину очереди
np.random.seed(42)
true_lambda = 2.5  # истинная интенсивность входящего потока заявок
true_mu = 3.0      # истинная интенсивность обработки заявок

num_events = 5000
timestamps = []
queue_sizes = []
events = []

current_time = 0.0
current_queue = 0

for _ in range(num_events):
    # общая интенсивность событий в состоянии current_queue
    if current_queue == 0:
        total_rate = true_lambda
    else:
        total_rate = true_lambda + true_mu

    # время до следующего события распределено экспоненциально (марковское свойство)
    dt = np.random.exponential(1.0 / total_rate)
    current_time += dt

    # определение типа события
    if current_queue == 0:
        event_type = 'arrival'
    else:
        # вероятность того, что произойдет именно приход заявки
        p_arrival = true_lambda / total_rate
        if np.random.rand() < p_arrival:
            event_type = 'arrival'
        else:
            event_type = 'departure'

    if event_type == 'arrival':
        current_queue += 1
    else:
        current_queue -= 1

    timestamps.append(current_time)
    queue_sizes.append(current_queue)
    events.append(event_type)

df_queue = pd.DataFrame({
    'timestamp': timestamps,
    'queue_size': queue_sizes,
    'event_type': events
})

print("фрагмент лога модификации очереди:")
print(df_queue.head(10))

фрагмент лога модификации очереди:
   timestamp  queue_size event_type
0   0.187707           1    arrival
1   0.735002           0  departure
2   1.100179           1    arrival
3   1.131020           2    arrival
4   1.141900           1  departure
5   1.309006           0  departure
6   1.317325           1    arrival
7   1.954336           0  departure
8   2.049811           1    arrival
9   2.086298           2    arrival


построение + обучение матрицы-генератора Q

In [14]:
# опрелеояем максимальный размер очереди в данных для задания размерности матрицы
max_observed_queue = df_queue['queue_size'].max()
K = max_observed_queue + 1

# для оценки интенсивностей нужно посчитать суммарное время нахождения системы в каждом состоянии и количество переходов из него
state_durations = np.zeros(K)
transition_counts = np.zeros((K, K))

# добавляем дельты по времени между модификациями
df_queue['dt'] = df_queue['timestamp'].diff().fillna(df_queue['timestamp'].iloc[0])

# итерирумеся по строкам для сбора статистики переходов
for i in range(len(df_queue) - 1):
    current_state = df_queue['queue_size'].iloc[i]
    next_state = df_queue['queue_size'].iloc[i+1]
    duration = df_queue['dt'].iloc[i+1]

    state_durations[current_state] += duration
    if current_state != next_state:
        transition_counts[current_state][next_state] += 1

# расчет инфинитезимальных коэффициентов q_ij = n_ij / T_i
Q = np.zeros((K, K))
for i in range(K):
    if state_durations[i] > 0:
        for j in range(K):
            if i != j:
                Q[i][j] = transition_counts[i][j] / state_durations[i]
        # диагональный элемент q_ii = минус сумма внедиагональных
        Q[i][i] = -np.sum(Q[i, :])

print("обученная генерационная матрица Q (фрагмент 5х5):")
print(pd.DataFrame(Q).iloc[:5, :5])

# извлекаем средние параметры лямбда и мю из матрицы для верификации
estimated_lambda = Q[1][2]
estimated_mu = Q[1][0]
print(f"\nоцененная интенсивность прихода (Lambda): {estimated_lambda:.3f} (истинная: {true_lambda})")
print(f"оцененная интенсивность обслуживания (Mu): {estimated_mu:.3f} (истинная: {true_mu})")

обученная генерационная матрица Q (фрагмент 5х5):
          0         1         2         3         4
0 -2.835940  2.835940  0.000000  0.000000  0.000000
1  3.114996 -5.842977  2.727981  0.000000  0.000000
2  0.000000  2.913845 -5.534281  2.620437  0.000000
3  0.000000  0.000000  3.095463 -5.806993  2.711530
4  0.000000  0.000000  0.000000  2.872173 -5.348624

оцененная интенсивность прихода (Lambda): 2.728 (истинная: 2.5)
оцененная интенсивность обслуживания (Mu): 3.115 (истинная: 3.0)


исследование адекватности модели (симуляция монте-карло)

In [15]:
def run_gillespie_simulation(Q_matrix, max_time=1000.0):
    """ симуляция траектории процесса по инфинитезимальной матрице """
    K_states = Q_matrix.shape[0]
    t = 0.0
    state = 0

    sim_times = [t]
    sim_states = [state]

    while t < max_time:
        q_ii = -Q_matrix[state][state]
        if q_ii == 0: # поглощающее состояние
            break

        # время до следующего скачка
        dt = np.random.exponential(1.0 / q_ii)
        t += dt

        # выбор следующего состояния
        probabilities = []
        possible_states = []
        for j in range(K_states):
            if j != state:
                probabilities.append(Q_matrix[state][j] / q_ii)
                possible_states.append(j)

        state = random.choices(possible_states, weights=probabilities)[0]

        sim_times.append(t)
        sim_states.append(state)

    return sim_times, sim_states

# запуск множества стохастических симуляций для усреднения результатов
num_simulations = 20
simulated_distributions = []

for _ in range(num_simulations):
    s_times, s_states = run_gillespie_simulation(Q, max_time=500.0)
    # считаем долю времени, проведенную в каждом состоянии
    durations = np.diff(s_times)
    states_until_end = s_states[:-1]

    dist = np.zeros(K)
    for s, d in zip(states_until_end, durations):
        dist[s] += d
    dist /= np.sum(durations)
    simulated_distributions.append(dist)

mean_simulated_dist = np.mean(simulated_distributions, axis=0)

# эмпирическое распределение из исходного датасета
empirical_dist = np.zeros(K)
for s, d in zip(df_queue['queue_size'], df_queue['dt']):
    empirical_dist[s] += d
empirical_dist /= df_queue['dt'].sum()

# сравниваем результаты
comparison_df = pd.DataFrame({
    'длина очереди': range(K),
    'эмпирический лог': empirical_dist,
    'усредненная симуляция Q': mean_simulated_dist
}).set_index('длина очереди')

print("сравнение стационарных распределений (дог vs симуляция):")
print(comparison_df.head(6))

# MAE по распределениям вероятностей
mae = np.mean(np.abs(empirical_dist - mean_simulated_dist))
print(f"\nсредняя абсолютная ошибка распределений (MAE): {mae:.5f}")
if mae < 0.05:
    print("сывод: различия минимальны. полученная инфинитезимальная модель адекватна исходным данным")
else:
    print("вывод: модель требует калибровки параметров")

сравнение стационарных распределений (дог vs симуляция):
               эмпирический лог  усредненная симуляция Q
длина очереди                                           
0                      0.058955                 0.132953
1                      0.176413                 0.119441
2                      0.101328                 0.114285
3                      0.095390                 0.092758
4                      0.073950                 0.085069
5                      0.073701                 0.070962

средняя абсолютная ошибка распределений (MAE): 0.00814
сывод: различия минимальны. полученная инфинитезимальная модель адекватна исходным данным
